In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('CIC_IDS_2017.csv')

C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_5808\1672948655.py:1: DtypeWarning: Columns (0,1,3,6,83) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('CIC_IDS_2017.csv')


In [3]:
df.duplicated().sum()

np.int64(288804)

In [4]:
global_dups = df.duplicated().sum()

label_dups = df.groupby("Label").apply(
    lambda x: x.duplicated().sum()
).sum()

print(global_dups, label_dups)


288804 203


C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_5808\3639644098.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  label_dups = df.groupby("Label").apply(


In [5]:
print(df["Label"].isna().sum())

288602


In [6]:
# Use dropna=False to include the rows with missing labels
df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())

C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_5808\2161749747.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())


Label
BENIGN                           202
Bot                                0
DDoS                               0
DoS GoldenEye                      0
DoS Hulk                           1
DoS Slowhttptest                   0
DoS slowloris                      0
FTP-Patator                        0
Heartbleed                         0
Infiltration                       0
PortScan                           0
SSH-Patator                        0
Web Attack – Brute Force           0
Web Attack – Sql Injection         0
Web Attack – XSS                   0
NaN                           288601
dtype: int64

In [7]:
df.dropna(subset=['Label'], inplace=True)

In [8]:
df.duplicated().sum()

np.int64(203)

In [9]:
df.shape

(2830743, 84)

In [10]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 1358


In [11]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 1358


In [12]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Byts/s']


In [13]:
df['Flow Byts/s'].isna().sum()

np.int64(1358)

In [14]:
# 1. Filter the dataframe for only rows where Flow Bytes/s is NaN
nan_data = df[df['Flow Byts/s'].isna()]

# 2. Count the labels within that subset
distribution = nan_data['Label'].value_counts()

print("Distribution of NaNs per Label:")
print(distribution)

# 3. Optional: See the percentage of each label that is "broken"
total_per_label = df['Label'].value_counts()
nan_percentage = (distribution / total_per_label) * 100
print("\nPercentage of each Label that has NaNs:")
print(nan_percentage.dropna())

Distribution of NaNs per Label:
Label
DoS Hulk    949
BENIGN      409
Name: count, dtype: int64

Percentage of each Label that has NaNs:
Label
BENIGN      0.017993
DoS Hulk    0.410693
Name: count, dtype: float64


In [15]:
df['Label'].value_counts()

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack – Brute Force         1507
Web Attack – XSS                  652
Infiltration                       36
Web Attack – Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [16]:
# 1. Select only the columns that contain text/objects
string_columns = df.select_dtypes(include=['object']).columns

# 2. Check if the stripped string is empty
# .str.strip() removes spaces; .eq('') checks if it is then empty
blanks_mask = df[string_columns].apply(lambda x: x.str.strip().eq('')).any(axis=1)

total_blank_rows = blanks_mask.sum()
print(f"Total rows with at least one blank entry: {total_blank_rows}")

Total rows with at least one blank entry: 0


In [17]:
import numpy as np

# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Total number of infinite values
total_inf = np.isinf(numeric_df).values.sum()

print(f"Total infinite entries: {total_inf}")

Total infinite entries: 4376


In [18]:
# Check if any value in a row is infinite
inf_rows = np.isinf(numeric_df).any(axis=1).sum()

print(f"Total rows with at least one infinite value: {inf_rows}")

Total rows with at least one infinite value: 2867


In [19]:
import numpy as np

# Select only numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Count infinite values (positive and negative) per column
inf_counts = np.isinf(df[numeric_cols]).sum()

# Display only columns that have at least one infinite value
print("Infinite values per column:")
print(inf_counts[inf_counts > 0])

Infinite values per column:
Flow Byts/s    1509
Flow Pkts/s    2867
dtype: int64


In [20]:
# 1. Create a boolean mask for any row that has at least one infinite value
inf_rows_mask = np.isinf(df[numeric_cols]).any(axis=1)

# 2. Filter the dataframe using that mask and count the Labels
inf_label_dist = df[inf_rows_mask]['Label'].value_counts()

print("\nDistribution of Infinite entries per Label:")
print(inf_label_dist)


Distribution of Infinite entries per Label:
Label
BENIGN         1777
DoS Hulk        949
PortScan        126
Bot              10
FTP-Patator       3
DDoS              2
Name: count, dtype: int64


In [21]:
total_inf_cells = np.isinf(df[numeric_cols]).values.sum()
total_rows_with_inf = inf_rows_mask.sum()

print(f"\nTotal infinite cells in dataset: {total_inf_cells}")
print(f"Total rows affected by infinity: {total_rows_with_inf}")


Total infinite cells in dataset: 4376
Total rows affected by infinity: 2867


In [22]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [23]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 2867


In [24]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 5734


In [25]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Byts/s', 'Flow Pkts/s']


In [26]:
# 1. Filter the dataframe for only rows where Flow Bytes/s is NaN
nan_data = df[df['Flow Byts/s'].isna()]

# 2. Count the labels within that subset
distribution = nan_data['Label'].value_counts()

print("Distribution of NaNs per Label:")
print(distribution)

# 3. Optional: See the percentage of each label that is "broken"
total_per_label = df['Label'].value_counts()
nan_percentage = (distribution / total_per_label) * 100
print("\nPercentage of each Label that has NaNs:")
print(nan_percentage.dropna())

Distribution of NaNs per Label:
Label
BENIGN         1777
DoS Hulk        949
PortScan        126
Bot              10
FTP-Patator       3
DDoS              2
Name: count, dtype: int64

Percentage of each Label that has NaNs:
Label
BENIGN         0.078175
Bot            0.508647
DDoS           0.001562
DoS Hulk       0.410693
FTP-Patator    0.037793
PortScan       0.079280
Name: count, dtype: float64


In [27]:
df.duplicated().sum()

np.int64(203)

In [28]:
df.describe()

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,...,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06
mean,4.112886e+04,8.071483e+03,9.880341e+00,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,...,5.418218e+00,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06
std,2.229494e+04,1.828363e+04,5.261922e+00,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,...,6.364257e+02,1.084989e+06,6.485999e+05,3.933815e+05,1.025825e+06,5.770923e+05,2.363008e+07,4.602984e+06,2.436689e+07,2.336342e+07
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.277400e+04,5.300000e+01,6.000000e+00,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.094400e+04,8.000000e+01,6.000000e+00,3.131600e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.841300e+04,4.430000e+02,1.700000e+01,3.204828e+06,5.000000e+00,4.000000e+00,1.870000e+02,4.820000e+02,8.100000e+01,3.600000e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,6.553500e+04,1.700000e+01,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,...,2.135570e+05,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [29]:
df.shape

(2830743, 84)

In [30]:
# 1. Create a copy to avoid modifying the original cleaned dataframe
binary_df = df.copy()

# 2. Use .map() or .apply() to transform the labels
# We check if the label is 'BENIGN' -> 0, otherwise -> 1
binary_df['Label'] = binary_df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# 3. Save to the new file
binary_df.to_csv('CIC-IDS-2017_Binary.csv', index=False)

print("Binary file created successfully!")
print(binary_df['Label'].value_counts())

Binary file created successfully!
Label
0    2273097
1     557646
Name: count, dtype: int64


In [31]:
benign_df = binary_df[binary_df['Label'] == 0]

In [32]:
benign_df.to_csv('Benign_Traffic_CIC2017.csv', index=False)

In [33]:
benign_df['Label'].value_counts()

Label
0    2273097
Name: count, dtype: int64

In [34]:
pd.set_option('display.max_rows', None)

# 2. Run your zero-count check again
# Replace 'df' with the name of your dataframe (e.g., X_normalized or data)
zero_counts = (df == 0).sum()

# 3. Print the result
print(zero_counts)

Flow ID                    0
Src IP                     0
Src Port                1696
Dst IP                     0
Dst Port                1696
Protocol                1696
Timestamp                  0
Flow Duration           2867
Tot Fwd Pkts               0
Tot Bwd Pkts          453519
TotLen Fwd Pkts       449170
TotLen Bwd Pkts       707746
Fwd Pkt Len Max       449170
Fwd Pkt Len Min      1317226
Fwd Pkt Len Mean      449170
Fwd Pkt Len Std      1850684
Bwd Pkt Len Max       707746
Bwd Pkt Len Min      1437867
Bwd Pkt Len Mean      707746
Bwd Pkt Len Std      2026514
Flow Byts/s           355767
Flow Pkts/s                0
Flow IAT Mean           2867
Flow IAT Std          998326
Flow IAT Max            2866
Flow IAT Min           75669
Fwd IAT Tot           711912
Fwd IAT Mean          711912
Fwd IAT Std          1776199
Fwd IAT Max           711912
Fwd IAT Min           743858
Bwd IAT Total        1249499
Bwd IAT Mean         1249499
Bwd IAT Std          2019228
Bwd IAT Max   

In [35]:
pd.set_option('display.max_rows', None)

# 2. Calculate the percentage of zeros
# (df_test_new == 0).mean() gives the proportion, * 100 gives the %
zero_percentage = (df == 0).mean() * 100

# 3. Print the result (sorted descending so you see the "emptiest" columns first)
print(zero_percentage)

Flow ID                0.000000
Src IP                 0.000000
Src Port               0.059914
Dst IP                 0.000000
Dst Port               0.059914
Protocol               0.059914
Timestamp              0.000000
Flow Duration          0.101281
Tot Fwd Pkts           0.000000
Tot Bwd Pkts          16.021200
TotLen Fwd Pkts       15.867566
TotLen Bwd Pkts       25.002128
Fwd Pkt Len Max       15.867566
Fwd Pkt Len Min       46.532871
Fwd Pkt Len Mean      15.867566
Fwd Pkt Len Std       65.378030
Bwd Pkt Len Max       25.002128
Bwd Pkt Len Min       50.794685
Bwd Pkt Len Mean      25.002128
Bwd Pkt Len Std       71.589473
Flow Byts/s           12.567972
Flow Pkts/s            0.000000
Flow IAT Mean          0.101281
Flow IAT Std          35.267278
Flow IAT Max           0.101246
Flow IAT Min           2.673114
Fwd IAT Tot           25.149298
Fwd IAT Mean          25.149298
Fwd IAT Std           62.746742
Fwd IAT Max           25.149298
Fwd IAT Min           26.277836
Bwd IAT 